# WavLM audio fine-tuning (Kaggle GPU step)

Fine-tunes `microsoft/wavlm-base-plus` (frozen CNN feature encoder, small classification head) on CREMA-D audio, **TRAIN actors only**, early-stopped on **VAL actors**' macro-F1, 3 seeds. Mirrors this project's standing rules: actor-level split (never touched here), macro-F1 primary metric, 3 seeds.

**Before training anything**, this notebook verifies that the public Kaggle CREMA-D audio mirror matches the project's local `AudioWAV` files exactly (file count, filenames, MD5 checksums) using a private dataset you attach containing `split_v1.json`, `excluded_clips.json`, `audio_md5_checksums.csv`, and a `supplementary_audio/` folder. **Known gap**: the `ejlok1/cremad` mirror is missing one canonical clip, `1040_ITH_SAD_XX.wav` -- it's supplied from `supplementary_audio/` instead and checksum-verified exactly like every other file. The verification report lists every problem found (missing, mismatched, extra) in one go; if anything's still wrong after checking the supplement, **it stops** -- it will not silently train on mismatched or substitute audio.

Public audio dataset: [kaggle.com/datasets/ejlok1/cremad](https://www.kaggle.com/datasets/ejlok1/cremad). Kaggle's mount depth for attached datasets varies (sometimes `/kaggle/input/<slug>/...`, sometimes `/kaggle/input/datasets/<owner>/<slug>/...`), so this notebook searches `/kaggle/input` **recursively** for `split_v1.json` (the private dataset) and for the directory containing the most `.wav` files (the audio mirror), and prints exactly what it found -- no fixed path is assumed. Either way, the checksum verification below is the real safety net, not the path.

**Test actors are never loaded for anything beyond the Step 1 checksum/count check** (which covers the full 7,438-clip canonical set so the same check is reusable later). All Step 2+ work (dataset construction, training, evaluation) is filtered to TRAIN/VAL actors only, with an explicit assertion. Test-set evaluation happens later, locally, per `FINAL_EVAL_PLAN.md` -- never here.

**Kaggle settings needed**: Settings -> Accelerator: GPU (T4 x2 or P100), Internet: On (to download the pretrained WavLM weights from Hugging Face).

In [ ]:
import sys, subprocess
for pkg in ["transformers", "soundfile", "torchaudio"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected -- check Settings > Accelerator before training.")

In [ ]:
import hashlib
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2FeatureExtractor, WavLMModel

CLASS_ORDER = ["anger", "disgust", "fear", "happy", "neutral", "sad"]
EMOTION_MAP = {"ANG": "anger", "DIS": "disgust", "FEA": "fear", "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
FILENAME_RE = re.compile(r"^(?P<actor_id>\d{4})_(?P<sentence>[A-Z]{3})_(?P<emotion>[A-Z]{3})_(?P<intensity>[A-Z]{2})$")
SEEDS = [0, 1, 2]
TARGET_SR = 16000
MAX_SECONDS = 6.0  # CREMA-D clips are mostly 2-5s; this is a safety ceiling, not a typical trim
WORKING_DIR = Path("/kaggle/working")

## Step 1: locate the attached datasets, then verify audio matches exactly

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
assert KAGGLE_INPUT.exists(), "This notebook expects to run on Kaggle with /kaggle/input mounted."

# Locate the private dataset by searching /kaggle/input RECURSIVELY for split_v1.json --
# Kaggle's mount depth varies (some datasets land at /kaggle/input/<slug>/, others at
# /kaggle/input/datasets/<owner>/<slug>/), so no fixed depth is assumed.
split_matches = list(KAGGLE_INPUT.rglob("split_v1.json"))
assert split_matches, (
    "Could not find split_v1.json anywhere under /kaggle/input (searched recursively). "
    "Attach the private dataset built from this project's kaggle_upload/ folder."
)
private_dir = split_matches[0].parent
print(f"private dataset found at: {private_dir}")
if len(split_matches) > 1:
    print(f"  (note: {len(split_matches)} copies of split_v1.json found under /kaggle/input; using the first: {split_matches[0]})")

# Locate the public CREMA-D audio mirror by searching /kaggle/input RECURSIVELY for
# .wav files and taking the directory with the most of them -- robust to whatever
# mount depth or internal folder layout the attached dataset uses. The checksum step
# below is the real safety net regardless of which directory this finds.
candidate_dirs: dict[Path, int] = {}
for wav_path in KAGGLE_INPUT.rglob("*.wav"):
    candidate_dirs[wav_path.parent] = candidate_dirs.get(wav_path.parent, 0) + 1
assert candidate_dirs, "No .wav files found anywhere under /kaggle/input -- attach the ejlok1/cremad dataset (or another CREMA-D audio mirror)."
audio_dir = max(candidate_dirs, key=candidate_dirs.get)
print(f"audio directory found at: {audio_dir} ({candidate_dirs[audio_dir]} .wav files)")
if len(candidate_dirs) > 1:
    print(f"  (other .wav-containing directories seen: { {str(k): v for k, v in candidate_dirs.items() if k != audio_dir} })")

In [ ]:
split_assignment = json.loads((private_dir / "split_v1.json").read_text())["assignment"]
excluded_info = json.loads((private_dir / "excluded_clips.json").read_text())
excluded_clip_ids = {c["clip_id"] for c in excluded_info["excluded_clips"]}
checksums_df = pd.read_csv(private_dir / "audio_md5_checksums.csv", dtype={"clip_id": str})
print(f"split actors: train={len(split_assignment['train'])} val={len(split_assignment['val'])} test={len(split_assignment['test'])}")
print(f"excluded clips: {sorted(excluded_clip_ids)}")
print(f"expected canonical clip count: {len(checksums_df)}")

In [ ]:
# --- VERIFICATION: count, filenames, checksums -- collects EVERY problem (missing,
# mismatched, extra) before stopping, rather than failing on the first one found. ---
kaggle_wav_files = {p.name: p for p in audio_dir.glob("*.wav")}
expected_filenames = set(checksums_df["filename"])

missing_on_kaggle = expected_filenames - set(kaggle_wav_files)
extra_on_kaggle = set(kaggle_wav_files) - expected_filenames  # informational -- the mirror may also include the 4 excluded raw files
print(f"local canonical count: {len(expected_filenames)} | kaggle (main mirror) found: {len(kaggle_wav_files)}")
print(f"missing from main mirror: {len(missing_on_kaggle)} | extra in main mirror: {len(extra_on_kaggle)}")

# Any canonical clip missing from the main mirror: look for it in the private dataset's
# supplementary_audio/ folder (searched recursively, so the exact internal layout doesn't matter).
from_supplement: dict[str, Path] = {}
if missing_on_kaggle:
    supplement_candidates = {p.name: p for p in private_dir.rglob("*.wav")}
    for filename in sorted(missing_on_kaggle):
        if filename in supplement_candidates:
            from_supplement[filename] = supplement_candidates[filename]
    for filename, path in from_supplement.items():
        kaggle_wav_files[filename] = path
    missing_on_kaggle = missing_on_kaggle - set(from_supplement)
    print(f"found {len(from_supplement)} missing clip(s) in the private dataset's supplementary_audio/: {sorted(from_supplement)}")

t0 = time.perf_counter()
mismatches = []
for _, row in checksums_df.iterrows():
    filename = row["filename"]
    if filename not in kaggle_wav_files:
        continue  # still missing -- already captured in missing_on_kaggle, don't double-report
    actual_md5 = hashlib.md5(kaggle_wav_files[filename].read_bytes()).hexdigest()
    if actual_md5 != row["md5"]:
        mismatches.append((filename, row["md5"], actual_md5))
print(f"checksummed {len(kaggle_wav_files)} available files in {time.perf_counter()-t0:.1f}s")

print("\n--- VERIFICATION REPORT ---")
print(f"still missing (not in main mirror or supplement): {len(missing_on_kaggle)}" + (f" -> {sorted(missing_on_kaggle)}" if missing_on_kaggle else ""))
print(f"checksum mismatches: {len(mismatches)}" + (f" -> {mismatches}" if mismatches else ""))
extra_preview = sorted(extra_on_kaggle)[:10]
print(f"extra files in main mirror (not fatal, informational): {len(extra_on_kaggle)}" + (f" -> {extra_preview}" if extra_on_kaggle else ""))
if from_supplement:
    print(f"supplied from the private dataset's supplementary_audio/: {sorted(from_supplement)}")

if missing_on_kaggle or mismatches:
    raise RuntimeError(
        f"STOPPING -- verification failed: {len(missing_on_kaggle)} clip(s) missing "
        f"(even after checking supplementary_audio/) and {len(mismatches)} checksum mismatch(es). "
        f"See the VERIFICATION REPORT printed above for the full list of every problem found. "
        f"Fix the dataset attachment (or add the missing clip(s) to kaggle_upload/supplementary_audio/ "
        f"via scripts/21_prepare_kaggle_upload.py) before re-running."
    )

print("\nVERIFICATION PASSED: Kaggle audio (main mirror + supplement) matches local files exactly (count, filenames, checksums).")

## Step 2: build the TRAIN/VAL dataframe (test actors excluded, asserted)

In [ ]:
train_actors = set(split_assignment["train"])
val_actors = set(split_assignment["val"])
test_actors = set(split_assignment["test"])

rows = []
for filename in expected_filenames:
    clip_id = filename[:-4]  # strip ".wav"
    if clip_id in excluded_clip_ids:
        continue
    m = FILENAME_RE.match(clip_id)
    if not m:
        raise RuntimeError(f"unexpected filename that doesn't match the CREMA-D convention: {filename}")
    actor_id, emotion_code = m.group("actor_id"), m.group("emotion")
    if emotion_code not in EMOTION_MAP:
        raise RuntimeError(f"unknown emotion code {emotion_code} in {filename}")
    rows.append({"clip_id": clip_id, "actor_id": actor_id, "emotion": EMOTION_MAP[emotion_code], "wav_path": str(kaggle_wav_files[filename])})

full_df = pd.DataFrame(rows)
print(f"canonical clips available: {len(full_df)}")

train_df = full_df[full_df["actor_id"].isin(train_actors)].reset_index(drop=True)
val_df = full_df[full_df["actor_id"].isin(val_actors)].reset_index(drop=True)

# Hard guarantee: this notebook never constructs a dataframe containing a test actor
# beyond this point. Test-set evaluation happens locally later, per FINAL_EVAL_PLAN.md.
assert not (set(train_df["actor_id"]) & test_actors)
assert not (set(val_df["actor_id"]) & test_actors)
assert not (set(train_df["actor_id"]) & set(val_df["actor_id"]))
print(f"train: {len(train_df)} clips, {train_df['actor_id'].nunique()} actors")
print(f"val:   {len(val_df)} clips, {val_df['actor_id'].nunique()} actors")
print(f"test actors excluded from this notebook entirely: {len(test_actors)} actors")

## Step 3: dataset, model (frozen CNN encoder + small head), training loop

In [ ]:
FEATURE_EXTRACTOR = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}


def load_waveform(path: str) -> np.ndarray:
    wav, sr = sf.read(path, dtype="float32", always_2d=False)
    if wav.ndim > 1:  # stereo -> mono
        wav = wav.mean(axis=1)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(torch.from_numpy(wav), sr, TARGET_SR).numpy()
    max_len = int(MAX_SECONDS * TARGET_SR)
    if len(wav) > max_len:
        wav = wav[:max_len]
    return wav


class AudioClipDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.rows = list(zip(df["wav_path"], df["emotion"], df["clip_id"]))

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int):
        path, emotion, clip_id = self.rows[i]
        wav = load_waveform(path)
        return wav, LABEL_TO_IDX[emotion], clip_id


def collate_batch(batch):
    wavs, labels, clip_ids = zip(*batch)
    processed = FEATURE_EXTRACTOR(list(wavs), sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    return processed["input_values"], processed["attention_mask"], torch.tensor(labels, dtype=torch.long), clip_ids

In [ ]:
class WavLMClassifier(nn.Module):
    """microsoft/wavlm-base-plus with its CNN feature encoder frozen, a
    masked-mean pool over the transformer's last hidden state, and a small
    2-layer classification head."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.backbone = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
        self.backbone.freeze_feature_encoder()  # freezes the conv feature encoder only
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(hidden // 2, num_classes))

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.backbone(input_values=input_values, attention_mask=attention_mask)
        feat_mask = self.backbone._get_feature_vector_attention_mask(out.last_hidden_state.shape[1], attention_mask)
        feat_mask = feat_mask.unsqueeze(-1).to(out.last_hidden_state.dtype)
        pooled = (out.last_hidden_state * feat_mask).sum(dim=1) / feat_mask.sum(dim=1).clamp(min=1.0)
        return self.head(pooled)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_clip_ids, all_true, all_pred, all_probs = [], [], [], []
    for input_values, attention_mask, labels, clip_ids in loader:
        input_values, attention_mask = input_values.to(device), attention_mask.to(device)
        logits = model(input_values, attention_mask)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_clip_ids.extend(clip_ids)
        all_true.extend(labels.numpy().tolist())
        all_pred.extend(preds.tolist())
        all_probs.extend(probs.tolist())
    y_true = [CLASS_ORDER[i] for i in all_true]
    y_pred = [CLASS_ORDER[i] for i in all_pred]
    macro_f1 = f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro")
    return macro_f1, all_clip_ids, y_true, y_pred, all_probs


def class_weights_for(df: pd.DataFrame) -> torch.Tensor:
    counts = df["emotion"].value_counts()
    n_total, n_classes = len(df), len(CLASS_ORDER)
    return torch.tensor([n_total / (n_classes * counts.get(c, 1)) for c in CLASS_ORDER], dtype=torch.float32)


def train_one_seed(train_df, val_df, seed: int, device, max_epochs=10, patience=3, batch_size=8, lr=1e-5, log_fn=print):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(AudioClipDataset(train_df), batch_size=batch_size, shuffle=True,
                               generator=torch.Generator().manual_seed(seed), collate_fn=collate_batch, num_workers=2)
    val_loader = DataLoader(AudioClipDataset(val_df), batch_size=batch_size * 2, shuffle=False, collate_fn=collate_batch, num_workers=2)

    model = WavLMClassifier(len(CLASS_ORDER)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(train_df).to(device))

    best_macro_f1, best_state, best_epoch, no_improve = -1.0, None, -1, 0
    history = []
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0.0
        for input_values, attention_mask, labels, _clip_ids in train_loader:
            input_values, attention_mask, labels = input_values.to(device), attention_mask.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(input_values, attention_mask), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        train_loss = total_loss / len(train_df)

        val_macro_f1, *_ = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_macro_f1": val_macro_f1})
        log_fn(f"  seed={seed} epoch={epoch} train_loss={train_loss:.4f} val_macro_f1={val_macro_f1:.4f}")

        if val_macro_f1 > best_macro_f1:
            best_macro_f1, best_epoch = val_macro_f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                log_fn(f"  seed={seed}: early stopping at epoch {epoch} (best epoch {best_epoch}, best val_macro_f1={best_macro_f1:.4f})")
                break

    model.load_state_dict(best_state)
    final_macro_f1, clip_ids, y_true, y_pred, probs = evaluate(model, val_loader, device)
    return {"model": model, "best_epoch": best_epoch, "history": history, "val_macro_f1": final_macro_f1,
            "clip_ids": clip_ids, "y_true": y_true, "y_pred": y_pred, "probs": probs}

## Step 4: run all 3 seeds, save per-clip VAL probabilities and checkpoints for download

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
WORKING_DIR.mkdir(exist_ok=True)

seed_results = []
all_prob_rows = []
for seed in SEEDS:
    print(f"\n{'='*70}\nSEED {seed}\n{'='*70}")
    result = train_one_seed(train_df, val_df, seed=seed, device=device)
    ckpt_path = WORKING_DIR / f"wavlm_base_plus_seed{seed}.pt"
    torch.save(result["model"].state_dict(), ckpt_path)
    print(f"seed={seed} FINAL val_macro_f1={result['val_macro_f1']:.4f} best_epoch={result['best_epoch']} -> {ckpt_path}")

    for clip_id, y_true, y_pred, probs in zip(result["clip_ids"], result["y_true"], result["y_pred"], result["probs"]):
        row = {"seed": seed, "clip_id": clip_id, "y_true": y_true, "y_pred": y_pred}
        row.update({f"prob_{c}": p for c, p in zip(CLASS_ORDER, probs)})
        all_prob_rows.append(row)

    seed_results.append({"seed": seed, "val_macro_f1": result["val_macro_f1"], "best_epoch": result["best_epoch"]})
    del result
    torch.cuda.empty_cache()

probs_df = pd.DataFrame(all_prob_rows)
probs_path = WORKING_DIR / "wavlm_val_probs.csv"
probs_df.to_csv(probs_path, index=False)
print(f"\nSaved per-clip VAL probabilities ({len(probs_df)} rows, {len(SEEDS)} seeds) to {probs_path}")

In [ ]:
summary_df = pd.DataFrame(seed_results)
summary_df.to_csv(WORKING_DIR / "wavlm_seed_summary.csv", index=False)
macro_f1s = summary_df["val_macro_f1"].to_numpy()
print(f"VAL macro-F1 across {len(SEEDS)} seeds: {macro_f1s.mean():.4f} +/- {macro_f1s.std():.4f} (per-seed: {[f'{v:.4f}' for v in macro_f1s]})")
print("\nFiles to download from /kaggle/working/ :")
for p in sorted(WORKING_DIR.glob("*")):
    print(f"  {p.name} ({p.stat().st_size / 1e6:.1f} MB)")